In [1]:
# %% [markdown]
# # Notebook 07 — Benchmark FAISS vs Index Séquentiel
#
# **Objectif** : Mesurer et comparer les performances de recherche entre
# l'ancien index séquentiel Python (O(n)) et le nouvel index FAISS,
# et valider que les résultats de recherche sont identiques.
#
# **Métriques mesurées** :
# - Throughput (req/s) selon la taille de l'index
# - Latence moyenne, p95, minimum
# - Cohérence des résultats (FAISS vs séquentiel)
# - Gain de scalabilité (facteur d'accélération)

# %%
import sys
import os
import time
import json
import tempfile
import warnings
import numpy  as np
import soundfile as sf

warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

import faiss
from backend.db.fingerprint_index    import FingerprintIndex
from backend.core.fingerprint_engine import FingerprintEngine
from backend.config                  import settings

print(f"FAISS version     : {faiss.__version__}")
print(f"FAISS GPU support : {faiss.get_num_gpus()} GPU(s)")

# %% [markdown]
# ## 0. Initialisation du moteur

# %%
engine = FingerprintEngine()
engine.load_grafprint_model()
print(f"Moteur : {'GraFPrint GNN' if engine.model else 'Fallback Librosa'}")
print(f"Dim empreinte : {engine.dim if hasattr(engine, 'dim') else 'auto'}")

FAISS version     : 1.14.3
FAISS GPU support : 0 GPU(s)
Moteur : GraFPrint GNN
Dim empreinte : auto


In [11]:
# %% [markdown]
# ## 1. Génération du corpus synthétique

# %%
import io
import tempfile
import os
import time
import numpy as np
import soundfile as sf

def make_audio_bytes(freq=440.0, sr=16000, dur=5.0, noise=0.02):
    """
    Génère un fichier audio en mémoire (bytes) sans laisser de fichiers temporaires.
    Utilise BytesIO pour éviter les problèmes de verrouillage Windows.
    """
    t = np.linspace(0, dur, int(sr * dur))
    s = np.sin(2 * np.pi * freq * t).astype(np.float32)
    s += noise * np.random.randn(len(s)).astype(np.float32)
    s /= np.abs(s).max()
    
    # Méthode propre : utiliser BytesIO au lieu d'un fichier temporaire
    buffer = io.BytesIO()
    sf.write(buffer, s, sr, format='wav')
    buffer.seek(0)
    return buffer.read()


# Version alternative avec gestion robuste des fichiers temporaires (si besoin)
def make_audio_bytes_with_cleanup(freq=440.0, sr=16000, dur=5.0, noise=0.02):
    """
    Version avec fichier temporaire mais nettoyage garanti.
    """
    t = np.linspace(0, dur, int(sr * dur))
    s = np.sin(2 * np.pi * freq * t).astype(np.float32)
    s += noise * np.random.randn(len(s)).astype(np.float32)
    s /= np.abs(s).max()
    
    tmp_path = None
    try:
        # Créer un fichier temporaire
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            sf.write(tmp.name, s, sr)
            tmp_path = tmp.name
        
        # Lire le fichier
        with open(tmp_path, "rb") as f:
            data = f.read()
        
        return data
        
    finally:
        # Nettoyage avec plusieurs tentatives
        if tmp_path and os.path.exists(tmp_path):
            for _ in range(5):
                try:
                    os.unlink(tmp_path)
                    break
                except PermissionError:
                    time.sleep(0.1)  # Attendre que le fichier soit libéré


# Tailles d'index à tester
SIZES = [10, 50, 100, 500, 1_000]
FREQS_BASE  = np.linspace(100, 8000, max(SIZES))
N_BENCH     = 30     # Itérations par mesure

print(f"Corpus : {max(SIZES)} fréquences de référence")
print(f"Benchmark : {N_BENCH} itérations par mesure")
print(f"Tailles testées : {SIZES}")

# Tester que la fonction fonctionne
test_bytes = make_audio_bytes(freq=440.0)
print(f"✅ Fonction make_audio_bytes OK - taille générée : {len(test_bytes)} bytes")

Corpus : 1000 fréquences de référence
Benchmark : 30 itérations par mesure
Tailles testées : [10, 50, 100, 500, 1000]
✅ Fonction make_audio_bytes OK - taille générée : 160044 bytes


In [12]:
# %% [markdown]
# ## 2. Génération des empreintes de référence

# %%
print("Génération des empreintes de référence...")
print("(Cette étape peut prendre quelques minutes selon le modèle)\n")

REFERENCE_EMBEDDINGS = []
REFERENCE_METADATA   = []

for i, freq in enumerate(FREQS_BASE[:max(SIZES)]):
    audio = make_audio_bytes(freq=freq, noise=0.01)
    emb   = engine.extract_fingerprint_from_bytes(audio)

    REFERENCE_EMBEDDINGS.append(emb)
    REFERENCE_METADATA.append({
        "title":  f"Oeuvre_{i+1:05d}",
        "artist": f"Artiste_{i % 10 + 1:02d}",
        "freq":   freq
    })

    if (i + 1) % 100 == 0 or (i + 1) == max(SIZES):
        print(f"  [{i+1}/{max(SIZES)}] Empreintes générées")

print(f"\n✅ {len(REFERENCE_EMBEDDINGS)} empreintes générées")
print(f"   Dim : {REFERENCE_EMBEDDINGS[0].shape[0]}")

# Requête de test (copie légèrement distordue de la première œuvre)
query_audio  = make_audio_bytes(freq=FREQS_BASE[0], noise=0.05)
QUERY_VECTOR = engine.extract_fingerprint_from_bytes(query_audio)
print(f"   Requête : Copie distordue de Oeuvre_00001 (freq={FREQS_BASE[0]:.0f}Hz, noise=0.05)")

Génération des empreintes de référence...
(Cette étape peut prendre quelques minutes selon le modèle)

  [100/1000] Empreintes générées
  [200/1000] Empreintes générées
  [300/1000] Empreintes générées
  [400/1000] Empreintes générées
  [500/1000] Empreintes générées
  [600/1000] Empreintes générées
  [700/1000] Empreintes générées
  [800/1000] Empreintes générées
  [900/1000] Empreintes générées
  [1000/1000] Empreintes générées

✅ 1000 empreintes générées
   Dim : 128
   Requête : Copie distordue de Oeuvre_00001 (freq=100Hz, noise=0.05)


In [13]:
# %% [markdown]
# ## 3. Index séquentiel (baseline O(n))

# %%
class SequentialIndex:
    """
    Index de référence (O(n)) — équivalent à l'implémentation avant FAISS.
    Utilisé uniquement pour le benchmark comparatif.
    """
    def __init__(self):
        self._data = {}

    def add(self, work_hash, embedding, meta):
        self._data[work_hash] = {"embedding": embedding, **meta}

    def search(self, query, top_k=1):
        results = []
        q_norm  = np.linalg.norm(query)
        if q_norm < 1e-8:
            return []
        for wh, entry in self._data.items():
            e     = np.array(entry["embedding"], dtype=np.float32)
            norm  = np.linalg.norm(e)
            if norm < 1e-8:
                continue
            score = float(np.dot(query, e) / (q_norm * norm))
            score = max(0.0, min(1.0, (score + 1.0) / 2.0))
            results.append({"work_hash": wh, "score": score, "title": entry["title"]})
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:top_k]

    def count(self):
        return len(self._data)


print("Index séquentiel (baseline) initialisé")

Index séquentiel (baseline) initialisé


In [14]:
# %% [markdown]
# ## 4. Benchmark comparatif

# %%
import hashlib

benchmark_results = {}

for size in SIZES:
    print(f"\n{'─' * 60}")
    print(f"  Taille d'index : {size} empreintes")
    print(f"{'─' * 60}")

    # ── Construction des deux index ────────────────────────────────────────
    tmp_dir  = tempfile.mkdtemp()
    tmp_path = os.path.join(tmp_dir, "bench_fingerprints.json")

    faiss_index = FingerprintIndex(settings.fingerprint_db.__class__(tmp_path))
    seq_index   = SequentialIndex()

    for i in range(size):
        emb  = REFERENCE_EMBEDDINGS[i]
        meta = REFERENCE_METADATA[i]
        wh   = hashlib.sha256(emb.tobytes()).hexdigest()

        faiss_index.add(
            embedding  = emb,
            title      = meta["title"],
            artist     = meta["artist"],
            ipfs_cid   = f"QmTest{i:05d}",
            tx_hash    = "0x" + "a" * 64,
            recipients = ["0x" + "A" * 40],
            shares     = [100]
        )
        seq_index.add(wh, emb, meta)

    print(f"  Index FAISS   : {faiss_index.count()} entrées | "
          f"type={faiss_index.stats()['index_type']}")
    print(f"  Index séquentiel : {seq_index.count()} entrées")

    # ── Benchmark FAISS ───────────────────────────────────────────────────
    faiss_times = []
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        faiss_index.search(QUERY_VECTOR, top_k=1)
        faiss_times.append((time.perf_counter() - t0) * 1000)

    # ── Benchmark séquentiel ──────────────────────────────────────────────
    seq_times = []
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        seq_index.search(QUERY_VECTOR, top_k=1)
        seq_times.append((time.perf_counter() - t0) * 1000)

    # ── Cohérence des résultats ───────────────────────────────────────────
    r_faiss = faiss_index.search(QUERY_VECTOR, top_k=1)
    r_seq   = seq_index.search(QUERY_VECTOR, top_k=1)

    faiss_score = r_faiss[0]["score"] if r_faiss else 0.0
    seq_score   = r_seq[0]["score"]   if r_seq   else 0.0
    coherent    = abs(faiss_score - seq_score) < 0.01

    # ── Calcul des métriques ──────────────────────────────────────────────
    faiss_arr  = np.array(faiss_times)
    seq_arr    = np.array(seq_times)
    speedup    = np.mean(seq_arr) / np.mean(faiss_arr)

    print(f"\n  FAISS   : mean={np.mean(faiss_arr):.3f}ms | "
          f"p95={np.percentile(faiss_arr, 95):.3f}ms | "
          f"{1000/np.mean(faiss_arr):.0f} req/s")
    print(f"  Séquentiel : mean={np.mean(seq_arr):.3f}ms | "
          f"p95={np.percentile(seq_arr, 95):.3f}ms | "
          f"{1000/np.mean(seq_arr):.0f} req/s")
    print(f"  Accélération : ×{speedup:.1f}")
    print(f"  Cohérence : {'✅ Scores identiques' if coherent else '⚠️ Scores divergents'} "
          f"(FAISS={faiss_score:.4f}, Séq={seq_score:.4f})")

    benchmark_results[size] = {
        "faiss_mean_ms":  np.mean(faiss_arr),
        "faiss_p95_ms":   np.percentile(faiss_arr, 95),
        "faiss_rps":      1000 / np.mean(faiss_arr),
        "seq_mean_ms":    np.mean(seq_arr),
        "seq_p95_ms":     np.percentile(seq_arr, 95),
        "seq_rps":        1000 / np.mean(seq_arr),
        "speedup":        speedup,
        "coherent":       coherent,
        "index_type":     faiss_index.stats()["index_type"]
    }

    # Nettoyage
    import shutil
    shutil.rmtree(tmp_dir, ignore_errors=True)

# %% [markdown]
# ## 5. Tableau comparatif final

# %%
print("\n" + "═" * 90)
print("  TABLEAU COMPARATIF — FAISS vs SÉQUENTIEL")
print("═" * 90)
print(f"\n  {'Taille':8s} {'Type index':22s} "
      f"{'FAISS (ms)':12s} {'Seq (ms)':12s} "
      f"{'FAISS (req/s)':15s} {'Seq (req/s)':13s} "
      f"{'Accél.':8s} {'Cohérence':10s}")
print("  " + "─" * 86)

for size, r in benchmark_results.items():
    print(
        f"  {size:8,d} "
        f"{r['index_type']:22s} "
        f"{r['faiss_mean_ms']:10.3f}   "
        f"{r['seq_mean_ms']:10.3f}   "
        f"{r['faiss_rps']:13,.0f}   "
        f"{r['seq_rps']:11,.0f}   "
        f"×{r['speedup']:5.1f}   "
        f"{'✅' if r['coherent'] else '❌':10s}"
    )

print(f"\n  Référence Chen et al. (2022) : 20 000 req/s (FAISS IVFPQ, GPU)")
print(f"  Notre implémentation (CPU)   : {benchmark_results[max(SIZES)]['faiss_rps']:,.0f} req/s "
      f"à {max(SIZES)} entrées")


────────────────────────────────────────────────────────────
  Taille d'index : 10 empreintes
────────────────────────────────────────────────────────────
  Index FAISS   : 10 entrées | type=FlatIP (exact)
  Index séquentiel : 10 entrées

  FAISS   : mean=1.713ms | p95=0.089ms | 584 req/s
  Séquentiel : mean=0.085ms | p95=0.114ms | 11697 req/s
  Accélération : ×0.0
  Cohérence : ✅ Scores identiques (FAISS=0.9885, Séq=0.9885)

────────────────────────────────────────────────────────────
  Taille d'index : 50 empreintes
────────────────────────────────────────────────────────────
  Index FAISS   : 50 entrées | type=FlatIP (exact)
  Index séquentiel : 50 entrées

  FAISS   : mean=0.052ms | p95=0.131ms | 19162 req/s
  Séquentiel : mean=0.306ms | p95=0.559ms | 3266 req/s
  Accélération : ×5.9
  Cohérence : ✅ Scores identiques (FAISS=0.9885, Séq=0.9885)

────────────────────────────────────────────────────────────
  Taille d'index : 100 empreintes
───────────────────────────────────────────

In [15]:
# %% [markdown]
# ## 6. Validation de la cohérence sur plusieurs requêtes

# %%
print("\n═" * 60)
print("  VALIDATION COHÉRENCE — 10 requêtes aléatoires")
print("═" * 60)

# Construire un index de taille 100
SIZE_VALID = 100
tmp_dir    = tempfile.mkdtemp()
tmp_path   = os.path.join(tmp_dir, "valid_fingerprints.json")

fi = FingerprintIndex(settings.fingerprint_db.__class__(tmp_path))
si = SequentialIndex()
wh_map = []

for i in range(SIZE_VALID):
    emb  = REFERENCE_EMBEDDINGS[i]
    meta = REFERENCE_METADATA[i]
    wh   = hashlib.sha256(emb.tobytes()).hexdigest()
    fi.add(emb, meta["title"], meta["artist"],
           f"Qm{i:05d}", "0x" + "a" * 64, ["0x" + "A" * 40], [100])
    si.add(wh, emb, meta)
    wh_map.append(wh)

# Requêtes sur des copies distorduites
print(f"\n  {'Requête':20s} {'FAISS score':14s} {'Seq score':12s} "
      f"{'Δ score':10s} {'Cohérent':10s}")
print("  " + "─" * 70)

n_coherent = 0
for j in range(10):
    src_idx = np.random.randint(0, SIZE_VALID)
    q_audio = make_audio_bytes(
        freq  = REFERENCE_METADATA[src_idx]["freq"],
        noise = np.random.uniform(0.01, 0.10)
    )
    q_emb = engine.extract_fingerprint_from_bytes(q_audio)

    rf = fi.search(q_emb, top_k=1)
    rs = si.search(q_emb, top_k=1)

    sf_score = rf[0]["score"] if rf else 0.0
    ss_score = rs[0]["score"] if rs else 0.0
    delta    = abs(sf_score - ss_score)
    ok       = delta < 0.02

    if ok:
        n_coherent += 1

    print(f"  Copie de {REFERENCE_METADATA[src_idx]['title']:12s} "
          f"{sf_score:.6f}     {ss_score:.6f}     "
          f"{delta:.6f}  {'✅' if ok else '❌'}")

print(f"\n  Cohérence globale : {n_coherent}/10 requêtes cohérentes")

import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
  VALIDATION COHÉRENCE — 10 requêtes aléatoires
════════════════════════════════════════════════════════════

  Requête              FAISS score    Seq score    Δ score    Cohérent  
  ──────────────────────────────────────────────────────────────────────
  Copie de Oeuvre_00065 0.993512     0.993512     0.000000  ✅
  Copie de Oeuvre_00065 0.990004     0.990004     0.000000  ✅
  Copie de Oeuvre_00075 0.996240     0.996240     0.000000  ✅
  Copie de Oeuvre_00068 0.980252     0.980252     0.000000  ✅
  Copie de Oeuvre_00032 0.981167     0.981167     0.000000  ✅
  Copie de Oeuvre_00066 0.983946     0.983946     0.000000  ✅
  Copie de Oeuvre_00019 0.999426     0.999426     0.000000  ✅
  Copie de Oeuvre_00062 0.975664     0.975664     0.000000  ✅
  Copie de Oeuvre_00018 0.991208     0.991208     0.000000  ✅
  Copie de Oeuvre_00013 0.985721     0.985721     0.000000  ✅

  

In [2]:
# %% [markdown]
# ## 7b. Test rapide avec tous les fichiers GTZAN réels

# %%
from pathlib import Path
import shutil

print("\n" + "═" * 60)
print("  TEST GTZAN — FAISS sur données réelles (TOUS les fichiers)")
print("═" * 60)

# Dossier racine des données
DATA_DIR = Path(PROJECT_ROOT) / "data"

# Créer un index FAISS temporaire
tmp_dir  = tempfile.mkdtemp()
tmp_path = Path(tmp_dir) / "gtzan_test.json"
gtzan_idx = FingerprintIndex(tmp_path)

# ── Charger TOUS les fichiers ──────────────────────────────────────────
ref_files    = sorted((DATA_DIR / "references").glob("*.wav"))
pirate_files = sorted((DATA_DIR / "pirates").glob("*.wav"))
legal_files  = sorted((DATA_DIR / "legal").glob("*.wav"))

print(f"\nFichiers trouvés :")
print(f"  Références : {len(ref_files)} → {[f.name for f in ref_files]}")
print(f"  Pirates    : {len(pirate_files)} → {[f.name for f in pirate_files]}")
print(f"  Légaux     : {len(legal_files)} → {[f.name for f in legal_files]}")

# ── Indexer toutes les références ──────────────────────────────────────
print("\nIndexation des œuvres de référence...")
for f in ref_files:
    emb = engine.extract_fingerprint(str(f))
    h   = FingerprintEngine.embedding_to_hash(emb)
    gtzan_idx.add(
        embedding  = emb,
        title      = f.stem,
        artist     = "GTZAN",
        ipfs_cid   = f"QmTest_{f.name}",
        tx_hash    = "0x" + "a" * 64,
        recipients = ["0x" + "A" * 40],
        shares     = [100]
    )
    print(f"  ✅ {f.name} — {len(emb)} dims")

print(f"\nIndex GTZAN : {gtzan_idx.count()} empreintes")

# ── Tester toutes les copies pirates ───────────────────────────────────
print("\n" + "─" * 60)
print("  RECHERCHE — Copies pirates (attendu : PIRATERIE)")
print("─" * 60)

for path in pirate_files:
    emb     = engine.extract_fingerprint(str(path))
    t0      = time.perf_counter()
    results = gtzan_idx.search(emb, top_k=1)
    dt      = (time.perf_counter() - t0) * 1000

    if results:
        r = results[0]
        detection = "🔴 PIRATERIE" if r['score'] >= 0.85 else "🟢 LÉGAL"
        correct   = "✅" if r['score'] >= 0.85 else "❌ FAUX NÉGATIF"
        print(f"  {path.name:35s} | score={r['score']:.4f} | {detection} | {correct} | {dt:.3f} ms")
    else:
        print(f"  {path.name:35s} | Aucun résultat")

# ── Tester tous les fichiers légaux ────────────────────────────────────
print("\n" + "─" * 60)
print("  RECHERCHE — Fichiers légaux (attendu : LÉGAL)")
print("─" * 60)

for path in legal_files:
    emb     = engine.extract_fingerprint(str(path))
    t0      = time.perf_counter()
    results = gtzan_idx.search(emb, top_k=1)
    dt      = (time.perf_counter() - t0) * 1000

    if results:
        r = results[0]
        detection = "🔴 PIRATERIE" if r['score'] >= 0.85 else "🟢 LÉGAL"
        correct   = "✅" if r['score'] < 0.85 else "❌ FAUX POSITIF"
        print(f"  {path.name:35s} | score={r['score']:.4f} | {detection} | {correct} | {dt:.3f} ms")
    else:
        print(f"  {path.name:35s} | Aucun résultat")

# ── Métriques globales ─────────────────────────────────────────────────
tp = sum(1 for p in pirate_files 
         if gtzan_idx.search(engine.extract_fingerprint(str(p)), top_k=1)[0]['score'] >= 0.85)
fn = len(pirate_files) - tp
fp = sum(1 for p in legal_files 
         if gtzan_idx.search(engine.extract_fingerprint(str(p)), top_k=1)[0]['score'] >= 0.85)
tn = len(legal_files) - fp

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

print(f"\n{'─' * 60}")
print(f"  MÉTRIQUES GLOBALES")
print(f"{'─' * 60}")
print(f"  TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print(f"  Précision : {precision:.4f}")
print(f"  Rappel    : {recall:.4f}")
print(f"  F1-score  : {f1:.4f}")

# ── Statistiques de l'index ────────────────────────────────────────────
print(f"\n  Index GTZAN : {gtzan_idx.count()} empreintes")
print(f"  Type        : {gtzan_idx.stats()['index_type']}")

# ── Nettoyage ──────────────────────────────────────────────────────────
shutil.rmtree(tmp_dir, ignore_errors=True)
print("\n✅ Test terminé — Index temporaire supprimé")


════════════════════════════════════════════════════════════
  TEST GTZAN — FAISS sur données réelles (TOUS les fichiers)
════════════════════════════════════════════════════════════

Fichiers trouvés :
  Références : 10 → ['blues.00001.wav', 'classical.00001.wav', 'country.00001.wav', 'disco.00008.wav', 'hiphop.00010.wav', 'jazz.00001.wav', 'metal.00001.wav', 'pop.00001.wav', 'reggae.00001.wav', 'rock.00001.wav']
  Pirates    : 10 → ['pirate_blues.00001.wav', 'pirate_classical.00001.wav', 'pirate_country.00001.wav', 'pirate_disco.00008.wav', 'pirate_hiphop.00010.wav', 'pirate_jazz.00001.wav', 'pirate_metal.00001.wav', 'pirate_pop.00001.wav', 'pirate_reggae.00001.wav', 'pirate_rock.00001.wav']
  Légaux     : 10 → ['blues.00005.wav', 'classical.00007.wav', 'country.00006.wav', 'disco.00012.wav', 'hiphop.00001.wav', 'jazz.00010.wav', 'metal.00011.wav', 'pop.00005.wav', 'reggae.00012.wav', 'rock.00013.wav']

Indexation des œuvres de référence...
  ✅ blues.00001.wav — 128 dims
  ✅ classic

In [17]:
# %% [markdown]
# ## 7. Test des statistiques d'index

# %%
print("\n═" * 60)
print("  STATISTIQUES DE L'INDEX PRINCIPAL")
print("═" * 60)

main_index = FingerprintIndex(settings.fingerprint_db)
stats      = main_index.stats()

for key, val in stats.items():
    print(f"  {key:20s} : {val}")

# %% [markdown]
# ## 8. Résumé

# %%
max_size = max(SIZES)
best_r   = benchmark_results[max_size]

print("\n" + "=" * 60)
print("  RÉSUMÉ NOTEBOOK 07 — Benchmark FAISS")
print("=" * 60)
print(f"  Tailles testées        : {SIZES}")
print(f"  Modèle                 : {'GraFPrint GNN' if engine.model else 'Fallback'}")
print(f"  Dim empreinte          : {REFERENCE_EMBEDDINGS[0].shape[0]}")
print(f"  FAISS à {max_size:,} entrées :")
print(f"    Type index           : {best_r['index_type']}")
print(f"    Latence moyenne      : {best_r['faiss_mean_ms']:.3f}ms")
print(f"    Throughput           : {best_r['faiss_rps']:,.0f} req/s")
print(f"    Accélération vs Seq  : ×{best_r['speedup']:.1f}")
print(f"    Cohérence résultats  : {'✅' if best_r['coherent'] else '❌'}")
print(f"  Réf. Chen et al. 2022  : 20 000 req/s (IVFPQ GPU)")
print("=" * 60)
print("  ✅ Notebook 07 validé — FAISS intégré et benchmarké")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
  STATISTIQUES DE L'INDEX PRINCIPAL
════════════════════════════════════════════════════════════
  count                : 3
  dim                  : 128
  index_type           : FlatIP (exact)
  faiss_total          : 3
  db_path              : c:\Users\Algis ADJINDA\music-antipiracy\backend\db\fingerprints.json
  faiss_path           : c:\Users\Algis ADJINDA\music-antipiracy\backend\db\fingerprints.faiss

  RÉSUMÉ NOTEBOOK 07 — Benchmark FAISS
  Tailles testées        : [10, 50, 100, 500, 1000]
  Modèle                 : GraFPrint GNN
  Dim empreinte          : 128
  FAISS à 1,000 entrées :
    Type index           : FlatIP (exact)
    Latence moyenne      : 0.071ms
    Throughput           : 14,101 req/s
    Accélération vs Seq  : ×118.1
    Cohérence résultats  : ✅
  Réf. Chen et al. 2022  : 20 000 req/s (IVFPQ GPU)
  ✅ Notebook 07 validé — FAISS intégré et benchm